<a href="https://colab.research.google.com/github/HeryTroop01/Computational_Methods/blob/main/Revit_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Python executable: /usr/bin/python3


In [2]:
import requests
import pydantic

print("requests:", requests.__version__)
print("pydantic:", pydantic.__version__)

requests: 2.32.4
pydantic: 2.13.5


In [3]:
SYSTEM_PROMPT = """
You are a command-generation assistant for a Windows Agent.

Your job is to convert the user's natural-language request
into a structured JSON command.

Available tools:

1. add_numbers
   Description: Add two numeric values.
   Parameters:
     - a: number
     - b: number

Rules:
- Return JSON only.
- Do not return Markdown.
- Do not return explanations.
- Use only the available tools.
- Never generate Python code.
- Never generate shell commands.

Example:

User:
Add 8 and 10

JSON:
{
  "command": "add_numbers",
  "parameters": {
    "a": 8,
    "b": 10
  }
}
"""


In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")


PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [7]:
!pip install -q transformers accelerate huggingface_hub


In [8]:
import transformers
import accelerate
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

Transformers: 5.16.1
Accelerate: 1.14.0
Hugging Face Hub: 1.29.0


In [9]:
from huggingface_hub import whoami

try:
    info = whoami()
    print("Logged in as:", info["name"])
except Exception:
    print("Not logged in to Hugging Face")


Not logged in to Hugging Face


In [12]:
import torch
import time

from transformers import AutoTokenizer, AutoModelForCausalLM


In [13]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print("Model:", MODEL_ID)

Model: Qwen/Qwen2.5-3B-Instruct


In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer loaded successfully")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully


In [15]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Model loaded successfully")
print("Device:", model.device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully
Device: cuda:0


In [16]:
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3

    print(f"GPU memory allocated: {allocated:.2f} GB")
    print(f"GPU memory reserved:  {reserved:.2f} GB")

GPU memory allocated: 5.75 GB
GPU memory reserved:  5.82 GB


In [18]:
SYSTEM_PROMPT = """
You are a command-generation assistant for a Windows Agent.

Your job is to convert the user's natural-language request
into a structured JSON command.

Available tools:

1. add_numbers
   Description: Add two numeric values.
   Parameters:
     - a: number
     - b: number

Rules:
- Return JSON only.
- Do not return Markdown.
- Do not return explanations.
- Use only the available tools.
- Never generate Python code.
- Never generate shell commands.

Example:

User:
Add 8 and 10

Output:
{
  "command": "add_numbers",
  "parameters": {
    "a": 8,
    "b": 10
  }
}
"""

USER_PROMPT = "Add 8 and 10"

In [19]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT,
    },
    {
        "role": "user",
        "content": USER_PROMPT,
    },
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(text)

<|im_start|>system

You are a command-generation assistant for a Windows Agent.

Your job is to convert the user's natural-language request
into a structured JSON command.

Available tools:

1. add_numbers
   Description: Add two numeric values.
   Parameters:
     - a: number
     - b: number

Rules:
- Return JSON only.
- Do not return Markdown.
- Do not return explanations.
- Use only the available tools.
- Never generate Python code.
- Never generate shell commands.

Example:

User:
Add 8 and 10

Output:
{
  "command": "add_numbers",
  "parameters": {
    "a": 8,
    "b": 10
  }
}
<|im_end|>
<|im_start|>user
Add 8 and 10<|im_end|>
<|im_start|>assistant



In [22]:
inputs = tokenizer(
    text,
    return_tensors="pt",
).to(model.device)

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
    )

elapsed = time.perf_counter() - start_time

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("LLM response:")
print(response)

print(f"\nGeneration time: {elapsed:.2f} seconds")

LLM response:
{
  "command": "add_numbers",
  "parameters": {
    "a": 8,
    "b": 10
  }
}

Generation time: 4.46 seconds


In [23]:
from pydantic import BaseModel, ConfigDict


class LLMCommand(BaseModel):
    model_config = ConfigDict(extra="forbid")

    command: str
    parameters: dict

In [24]:
import json

parsed = json.loads(response)

print(parsed)

{'command': 'add_numbers', 'parameters': {'a': 8, 'b': 10}}


In [25]:
command = LLMCommand.model_validate(parsed)

print(command)

command='add_numbers' parameters={'a': 8, 'b': 10}


In [26]:
clean_command = command.model_dump()

print(json.dumps(clean_command, indent=2))

{
  "command": "add_numbers",
  "parameters": {
    "a": 8,
    "b": 10
  }
}


In [28]:
import json
import time
from pydantic import BaseModel, ConfigDict


class LLMCommand(BaseModel):
    model_config = ConfigDict(extra="forbid")

    command: str
    parameters: dict


def generate_command(user_prompt):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    start = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
        )

    elapsed = time.perf_counter() - start

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    raw_response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    parsed = json.loads(raw_response)
    validated = LLMCommand.model_validate(parsed)

    return validated, raw_response, elapsed

In [31]:
command, raw, elapsed = generate_command(
    "Create a wall that is 5 meters long"
)

print("Raw response:")
print(raw)

print("\nValidated command:")
print(command.model_dump())

print(f"\nGeneration time: {elapsed:.2f} seconds")

Raw response:
{
  "command": "create_wall",
  "parameters": {
    "length": 5,
    "unit": "meters"
  }
}

Validated command:
{'command': 'create_wall', 'parameters': {'length': 5, 'unit': 'meters'}}

Generation time: 4.28 seconds


In [32]:
TOOLS = [
    {
        "name": "add_numbers",
        "description": "Add two numeric values.",
        "parameters": {
            "a": "number",
            "b": "number"
        }
    }
]

print(TOOLS)

[{'name': 'add_numbers', 'description': 'Add two numeric values.', 'parameters': {'a': 'number', 'b': 'number'}}]


In [33]:
def build_tool_description(tools):
    lines = []

    for tool in tools:
        lines.append(f"Tool: {tool['name']}")
        lines.append(f"Description: {tool['description']}")
        lines.append("Parameters:")

        for name, parameter_type in tool["parameters"].items():
            lines.append(f"  - {name}: {parameter_type}")

        lines.append("")

    return "\n".join(lines)


tool_description = build_tool_description(TOOLS)

print(tool_description)

Tool: add_numbers
Description: Add two numeric values.
Parameters:
  - a: number
  - b: number



In [35]:
TOOL_AWARE_SYSTEM_PROMPT = f"""
You are a command-generation assistant for a Windows Agent.

Your task is to convert the user's natural-language request
into a structured JSON command.

These are the ONLY tools currently available:

{tool_description}

Rules:

1. Return JSON only.
2. Do not return Markdown.
3. Do not return explanations.
4. Use only tools listed above.
5. Never invent a tool.
6. Never generate Python code.
7. Never generate PowerShell commands.
8. The command field must contain an available tool name.
9. The parameters field must contain the parameters required by that tool.

If the user's request cannot be fulfilled by an available tool,
return:

{{
  "command": "unsupported",
  "parameters": {{}}
}}

Example:

User:
Add 8 and 10

Output:
{{
  "command": "add_numbers",
  "parameters": {{
    "a": 8,
    "b": 10
  }}
}}
"""

In [36]:
def generate_tool_aware_command(user_prompt):
    messages = [
        {
            "role": "system",
            "content": TOOL_AWARE_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    start = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
        )

    elapsed = time.perf_counter() - start

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    raw_response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    parsed = json.loads(raw_response)

    return parsed, raw_response, elapsed

In [38]:
result, raw, elapsed = generate_tool_aware_command(
    "Add 8 and 10"
)

print("Raw:")
print(raw)

print("\nParsed:")
print(json.dumps(result, indent=2))

print(f"\nGeneration time: {elapsed:.2f} seconds")

Raw:
{
  "command": "add_numbers",
  "parameters": {
    "a": 8,
    "b": 10
  }
}

Parsed:
{
  "command": "add_numbers",
  "parameters": {
    "a": 8,
    "b": 10
  }
}

Generation time: 1.79 seconds


In [40]:
def validate_command(command, tools):
    available_names = {tool["name"] for tool in tools}

    if command["command"] == "unsupported":
        return False, "Request cannot be fulfilled by available tools."

    if command["command"] not in available_names:
        return False, f"Unknown tool generated: {command['command']}"

    return True, "Command is valid."

In [41]:
valid, message = validate_command(
    {
        "command": "add_numbers",
        "parameters": {
            "a": 8,
            "b": 10
        }
    },
    TOOLS
)

print("Valid:", valid)
print("Message:", message)

Valid: True
Message: Command is valid.


In [42]:
valid, message = validate_command(
    {
        "command": "create_wall",
        "parameters": {
            "length": 5
        }
    },
    TOOLS
)

print("Valid:", valid)
print("Message:", message)

Valid: False
Message: Unknown tool generated: create_wall


In [43]:
WINDOWS_AGENT_URL = "https://sender-grove-skirt-occupational.trycloudflare.com"

In [65]:
import requests

url = f"{WINDOWS_AGENT_URL}/health"

response = requests.get(
    url,
    timeout=10,
)

print("HTTP status:", response.status_code)
print("Response:", response.json())

HTTP status: 200
Response: {'status': 'healthy', 'protocol_version': '1.0'}


In [66]:
response = requests.get(
    f"{WINDOWS_AGENT_URL}/tools",
    timeout=10,
)

print("HTTP status:", response.status_code)

tools_response = response.json()

print(json.dumps(tools_response, indent=2))

HTTP status: 200
{
  "protocol_version": "1.0",
  "tools": [
    {
      "name": "ping",
      "description": "Test whether the Windows Agent tool system is working.",
      "enabled": true,
      "parameters": {
        "additionalProperties": false,
        "properties": {},
        "title": "EmptyParameters",
        "type": "object"
      }
    },
    {
      "name": "add_numbers",
      "description": "Add two numeric values.",
      "enabled": true,
      "parameters": {
        "additionalProperties": false,
        "properties": {
          "a": {
            "title": "A",
            "type": "number"
          },
          "b": {
            "title": "B",
            "type": "number"
          }
        },
        "required": [
          "a",
          "b"
        ],
        "title": "AddNumbersParameters",
        "type": "object"
      }
    }
  ]
}


In [67]:
WINDOWS_TOOLS = tools_response["tools"]

for tool in WINDOWS_TOOLS:
    print(
        f"{tool['name']} → "
        f"{tool['description']} "
        f"[enabled={tool['enabled']}]"
    )

ping → Test whether the Windows Agent tool system is working. [enabled=True]
add_numbers → Add two numeric values. [enabled=True]


In [47]:
LLM_TOOLS = []

for tool in WINDOWS_TOOLS:
    if tool["enabled"]:
        LLM_TOOLS.append({
            "name": tool["name"],
            "description": tool["description"],
        })

print(json.dumps(LLM_TOOLS, indent=2))

[
  {
    "name": "ping",
    "description": "Test whether the Windows Agent tool system is working."
  },
  {
    "name": "add_numbers",
    "description": "Add two numeric values."
  }
]


In [49]:
tools_response = requests.get(
    f"{WINDOWS_AGENT_URL}/tools",
    timeout=10,
).json()

WINDOWS_TOOLS = tools_response["tools"]

print(json.dumps(WINDOWS_TOOLS, indent=2))

[
  {
    "name": "ping",
    "description": "Test whether the Windows Agent tool system is working.",
    "enabled": true
  },
  {
    "name": "add_numbers",
    "description": "Add two numeric values.",
    "enabled": true
  }
]


In [50]:
def build_live_tool_description(tools):
    lines = []

    for tool in tools:
        if not tool["enabled"]:
            continue

        lines.append(
            f"Tool: {tool['name']}\n"
            f"Description: {tool['description']}"
        )

    return "\n\n".join(lines)


live_tool_description = build_live_tool_description(
    WINDOWS_TOOLS
)

print(live_tool_description)

Tool: ping
Description: Test whether the Windows Agent tool system is working.

Tool: add_numbers
Description: Add two numeric values.


In [76]:
def build_live_tool_description(tools):
    lines = []

    for tool in tools:
        if not tool["enabled"]:
            continue

        lines.append(f"Tool: {tool['name']}")
        lines.append(f"Description: {tool['description']}")

        schema = tool.get("parameters", {})
        properties = schema.get("properties", {})
        required = schema.get("required", [])

        lines.append("Parameters:")

        if not properties:
            lines.append("  None")
        else:
            for name, info in properties.items():
                parameter_type = info.get("type", "unknown")
                required_text = "required" if name in required else "optional"

                lines.append(
                    f"  - {name}: {parameter_type} ({required_text})"
                )

        lines.append("")

    return "\n".join(lines)


live_tool_description = build_live_tool_description(
    WINDOWS_TOOLS
)

print(live_tool_description)

Tool: ping
Description: Test whether the Windows Agent tool system is working.
Parameters:
  None

Tool: add_numbers
Description: Add two numeric values.
Parameters:
  - a: number (required)
  - b: number (required)



In [89]:
LIVE_SYSTEM_PROMPT = f"""
You are the command-generation component of a Windows Agent system.

Convert the user's natural-language request into a JSON command.

The Windows Agent has provided these currently available tools:

{live_tool_description}

Rules:

1. Return JSON only.
2. Do not return Markdown.
3. Do not return explanations.
4. Use only tools listed above.
5. Never invent a tool.
6. Never generate Python code.
7. Never generate shell commands.
8. The command must be an available tool name.
9. Use the exact parameter names provided by the tool schema.
10. Do not rename parameters.
11. If no available tool can perform the request, return:

{{
  "command": "unsupported",
  "parameters": {{}}
}}

Example:

User:
Add 8 and 10

Output:
{{
  "command": "add_numbers",
  "parameters": {{
    "a": 8,
    "b": 10
  }}
}}
"""

In [78]:
def generate_live_command(user_prompt):
    messages = [
        {
            "role": "system",
            "content": LIVE_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    start = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
        )

    elapsed = time.perf_counter() - start

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    raw_response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    parsed = json.loads(raw_response)

    return parsed, raw_response, elapsed

In [90]:
result, raw, elapsed = generate_live_command(
    "Add 8 and 10"
)

print("Raw LLM response:")
print(raw)

print("\nParsed command:")
print(json.dumps(result, indent=2))

print(f"\nGeneration time: {elapsed:.2f} seconds")

Raw LLM response:
{
  "command": "add_numbers",
  "parameters": {
    "a": 8,
    "b": 10
  }
}

Parsed command:
{
  "command": "add_numbers",
  "parameters": {
    "a": 8,
    "b": 10
  }
}

Generation time: 1.79 seconds


In [80]:
available_commands = {
    tool["name"]
    for tool in WINDOWS_TOOLS
    if tool["enabled"]
}

print("Available commands:")
print(available_commands)

print("\nLLM command:")
print(result["command"])

if result["command"] in available_commands:
    print("\nCommand validation: PASS")
else:
    print("\nCommand validation: FAIL")

Available commands:
{'ping', 'add_numbers'}

LLM command:
add_numbers

Command validation: PASS


In [81]:
result, raw, elapsed = generate_live_command(
    "Create a wall that is 5 meters long"
)

print("Raw response:")
print(raw)

print("\nParsed command:")
print(json.dumps(result, indent=2))

print(f"\nGeneration time: {elapsed:.2f} seconds")

Raw response:
{
  "command": "unsupported",
  "parameters": {}
}

Parsed command:
{
  "command": "unsupported",
  "parameters": {}
}

Generation time: 0.84 seconds


In [82]:
def send_command_to_windows(command):
    response = requests.post(
        f"{WINDOWS_AGENT_URL}/command",
        json=command,
        timeout=10,
    )

    print("HTTP status:", response.status_code)

    response.raise_for_status()

    return response.json()

In [83]:
result, raw, elapsed = generate_live_command(
    "Add 8 and 10"
)

print("LLM output:")
print(raw)

print("\nCommand:")
print(json.dumps(result, indent=2))

print(f"\nLLM generation time: {elapsed:.2f} seconds")

LLM output:
{
  "command": "add_numbers",
  "parameters": {
    "a": 8,
    "b": 10
  }
}

Command:
{
  "command": "add_numbers",
  "parameters": {
    "a": 8,
    "b": 10
  }
}

LLM generation time: 1.76 seconds


In [84]:
available_commands = {
    tool["name"]
    for tool in WINDOWS_TOOLS
    if tool["enabled"]
}

if result["command"] not in available_commands:
    raise ValueError(
        f"LLM generated unavailable command: {result['command']}"
    )

print("Command validation: PASS")

Command validation: PASS


In [85]:
windows_result = send_command_to_windows(result)

print(json.dumps(windows_result, indent=2))

HTTP status: 200
{
  "success": true,
  "protocol_version": "1.0",
  "command": "add_numbers",
  "result": {
    "result": 18.0
  },
  "error": null
}


In [86]:
import requests
import json

tools_response = requests.get(
    f"{WINDOWS_AGENT_URL}/tools",
    timeout=10,
).json()

WINDOWS_TOOLS = tools_response["tools"]

print(json.dumps(WINDOWS_TOOLS, indent=2))


[
  {
    "name": "ping",
    "description": "Test whether the Windows Agent tool system is working.",
    "enabled": true,
    "parameters": {
      "additionalProperties": false,
      "properties": {},
      "title": "EmptyParameters",
      "type": "object"
    }
  },
  {
    "name": "add_numbers",
    "description": "Add two numeric values.",
    "enabled": true,
    "parameters": {
      "additionalProperties": false,
      "properties": {
        "a": {
          "title": "A",
          "type": "number"
        },
        "b": {
          "title": "B",
          "type": "number"
        }
      },
      "required": [
        "a",
        "b"
      ],
      "title": "AddNumbersParameters",
      "type": "object"
    }
  }
]


In [88]:
print(live_tool_description)

Tool: ping
Description: Test whether the Windows Agent tool system is working.
Parameters:
  None

Tool: add_numbers
Description: Add two numeric values.
Parameters:
  - a: number (required)
  - b: number (required)



In [91]:
tool = next(
    t for t in WINDOWS_TOOLS
    if t["name"] == result["command"]
)

schema = tool["parameters"]

print("Tool:", tool["name"])
print("Required parameters:", schema.get("required"))
print("Received parameters:", list(result["parameters"].keys()))

Tool: add_numbers
Required parameters: ['a', 'b']
Received parameters: ['a', 'b']


In [93]:
def validate_against_tool_schema(command, tools):
    tool = next(
        (
            t for t in tools
            if t["name"] == command["command"]
            and t["enabled"]
        ),
        None
    )

    if tool is None:
        return False, f"Tool not available: {command['command']}"

    schema = tool.get("parameters", {})
    properties = schema.get("properties", {})
    required = schema.get("required", [])

    parameters = command.get("parameters", {})

    missing = [
        name for name in required
        if name not in parameters
    ]

    unknown = [
        name for name in parameters
        if name not in properties
    ]

    if missing:
        return False, f"Missing parameters: {missing}"

    if unknown:
        return False, f"Unknown parameters: {unknown}"

    return True, "Schema validation passed"

In [94]:
valid, message = validate_against_tool_schema(
    result,
    WINDOWS_TOOLS
)

print("Valid:", valid)
print("Message:", message)

Valid: True
Message: Schema validation passed


In [95]:
def send_command_to_windows(command):
    response = requests.post(
        f"{WINDOWS_AGENT_URL}/command",
        json=command,
        timeout=10,
    )

    print("HTTP status:", response.status_code)

    response.raise_for_status()

    return response.json()


In [96]:
user_request = "Add 8 and 10"

command, raw, llm_time = generate_live_command(
    user_request
)

print("User request:")
print(user_request)

print("\nLLM output:")
print(raw)

print("\nParsed command:")
print(json.dumps(command, indent=2))

print(f"\nGeneration time: {llm_time:.2f} seconds")

User request:
Add 8 and 10

LLM output:
{
  "command": "add_numbers",
  "parameters": {
    "a": 8,
    "b": 10
  }
}

Parsed command:
{
  "command": "add_numbers",
  "parameters": {
    "a": 8,
    "b": 10
  }
}

Generation time: 1.79 seconds


In [97]:
valid, message = validate_against_tool_schema(
    command,
    WINDOWS_TOOLS
)

print("Validation:", valid)
print("Message:", message)

Validation: True
Message: Schema validation passed


In [98]:
windows_result = send_command_to_windows(command)

print("\nWindows Agent response:")
print(json.dumps(windows_result, indent=2))

HTTP status: 200

Windows Agent response:
{
  "success": true,
  "protocol_version": "1.0",
  "command": "add_numbers",
  "result": {
    "result": 18.0
  },
  "error": null
}


In [99]:
user_request = "Add 8 and 10"

print("=" * 60)
print("USER REQUEST")
print("=" * 60)
print(user_request)

# Generate command
command, raw, llm_time = generate_live_command(
    user_request
)

print("\n" + "=" * 60)
print("LLM OUTPUT")
print("=" * 60)
print(raw)

# Validate command
valid, message = validate_against_tool_schema(
    command,
    WINDOWS_TOOLS
)

print("\n" + "=" * 60)
print("SCHEMA VALIDATION")
print("=" * 60)
print(message)

if not valid:
    raise ValueError(message)

# Execute on Windows
windows_result = send_command_to_windows(command)

print("\n" + "=" * 60)
print("WINDOWS AGENT")
print("=" * 60)
print(json.dumps(windows_result, indent=2))

print("\n" + "=" * 60)
print("TIMING")
print("=" * 60)
print(f"LLM generation: {llm_time:.2f} seconds")

USER REQUEST
Add 8 and 10

LLM OUTPUT
{
  "command": "add_numbers",
  "parameters": {
    "a": 8,
    "b": 10
  }
}

SCHEMA VALIDATION
Schema validation passed
HTTP status: 200

WINDOWS AGENT
{
  "success": true,
  "protocol_version": "1.0",
  "command": "add_numbers",
  "result": {
    "result": 18.0
  },
  "error": null
}

TIMING
LLM generation: 2.07 seconds
